# PAN 2026 Proxy Evaluation — BM25 Baseline

Same dataset and evaluation as `pan2026_proxy_eval.ipynb` but using BM25 (PyTerrier) instead of e5+FAISS.

Reads directly from the exported dataset (`pan2026_proxy_dataset.zip`):
- `corpus.jsonl.gz` — source chunks (the pool to retrieve from)
- `queries.jsonl` — query chunks (plagiarized spans from suspicious docs)
- `qrels.txt` — ground truth relevance (TREC format)

Pipeline: BM25 (PyTerrier) → Recall@1 at query chunk level

In [ ]:
!pip install python-terrier -q

from google.colab import drive
drive.mount('/content/drive')

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.8/208.8 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 3.2 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import shutil
import subprocess

print("Copying dataset zip from Drive...")
shutil.copy2(
    "/content/drive/MyDrive/pan2026_proxy_dataset.zip",
    "/content/pan2026_proxy_dataset.zip"
)
print("Unzipping...")
subprocess.run(["unzip", "-q", "/content/pan2026_proxy_dataset.zip", "-d", "/content/pan2026_proxy_dataset/"])
print("Done!")

Copying dataset zip from Drive...
Unzipping...
Done!


## Imports & Config

In [ ]:
import json
import gzip
import pandas as pd
import pyterrier as pt
from pathlib import Path

if not pt.started():
    pt.init()

terrier-assemblies 5.11 jar-with-dependencies not found, downloading to /root/.pyterrier...


/tmp/ipykernel_962/3371630599.py:7: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():


https://repo1.maven.org/maven2/org/terrier/terrier-assemblies/5.11/terrier-assemblies-5.11-jar-with-dependenci…

Done
terrier-python-helper 0.0.8 jar not found, downloading to /root/.pyterrier...


https://repo1.maven.org/maven2/org/terrier/terrier-python-helper/0.0.8/terrier-python-helper-0.0.8.jar:   0%| …

Done


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_962/3371630599.py:8: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [ ]:
DATASET_DIR  = Path("/content/pan2026_proxy_dataset")
CORPUS_PATH  = DATASET_DIR / "corpus.jsonl.gz"
QUERIES_PATH = DATASET_DIR / "queries.jsonl"
QRELS_PATH   = DATASET_DIR / "qrels.txt"

# index stored on Drive so it persists across sessions
INDEX_DIR = Path("/content/drive/MyDrive/pan2026_bm25_index")

## Load Dataset

In [ ]:
# Load corpus
print("Loading corpus...")
corpus = []
with gzip.open(CORPUS_PATH, "rt", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        corpus.append({"docno": obj["doc_id"], "text": obj["default_text"]})
print(f"Corpus: {len(corpus)} source chunks")

# Load queries
print("Loading queries...")
queries = []
with open(QUERIES_PATH, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        queries.append({"qid": obj["qid"], "query": obj["query"]})
print(f"Queries: {len(queries)} query chunks")

# Load qrels
print("Loading qrels...")
qrels = {}
with open(QRELS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 4:
            qid, _, doc_id, rel = parts
            if int(rel) > 0:
                qrels.setdefault(qid, set()).add(doc_id)
print(f"Qrels: {len(qrels)} queries have relevance judgements")

Loading corpus...
Corpus: 13705 source chunks
Loading queries...
Queries: 4676 query chunks
Loading qrels...
Qrels: 4627 queries have relevance judgements


## Build BM25 Index

In [ ]:
INDEX_DIR.mkdir(parents=True, exist_ok=True)
index_path = INDEX_DIR.resolve().absolute()

if not (index_path / "data.properties").exists():
    print("Building index...")
    indexer = pt.IterDictIndexer(
        str(index_path),
        overwrite=True,
        meta={"docno": 100, "text": 20480}
    )
    indexer.index(iter(corpus))
    print("Index built!")
else:
    print("Index already exists — loading")

index = pt.IndexFactory.of(str(index_path))
bm25  = pt.terrier.Retriever(index, wmodel="BM25", num_results=1)

Building index...
17:50:49.711 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Adding an empty document to the index (source-document014670_para_151) - further warnings are suppressed
17:50:56.299 [ForkJoinPool-1-worker-1] WARN org.terrier.structures.indexing.Indexer -- Indexed 2 empty documents
Index built!


## Retrieval & Evaluation

In [ ]:
print("Tokenising queries...")
tokeniser = pt.java.autoclass(
    "org.terrier.indexing.tokenisation.Tokeniser"
).getTokeniser()

topics = pd.DataFrame(queries)
topics["query"] = topics["query"].apply(
    lambda q: " ".join(tokeniser.getTokens(q))
)

print("Retrieving top-1 for each query chunk...")
run = bm25(topics)

# Evaluate: Recall@1 at query chunk level
tp = 0
fp = 0
fn = 0
results = []

for _, row in run.iterrows():
    qid          = row["qid"]
    retrieved_id = row["docno"]
    score        = row["score"]
    relevant     = qrels.get(qid, set())

    hit = retrieved_id in relevant
    if hit:
        tp += 1
    else:
        fp += 1
        fn += 1

    results.append({
        "qid":        qid,
        "retrieved":  retrieved_id,
        "score":      round(float(score), 4),
        "hit":        hit,
        "n_relevant": len(relevant),
    })

recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

print("\n── Results ──────────────────────────────────────────")
print(f"  Recall@1 : {round(recall, 3)}")
print(f"  TP       : {tp}")
print(f"  FP       : {fp}")
print(f"  FN       : {fn}")
print(f"  Evaluated: {tp + fp} query chunks")
print("─────────────────────────────────────────────────────")

Tokenising queries...
Retrieving top-1 for each query chunk...

── Results ──────────────────────────────────────────
  Recall@1 : 0.675
  TP       : 3222
  FP       : 1552
  FN       : 1552
  Evaluated: 4774 query chunks
─────────────────────────────────────────────────────


## Inspect Hits & Misses

In [ ]:
corpus_text_map = {doc["docno"]: doc["text"] for doc in corpus}
query_text_map  = {q["qid"]: q["query"]     for q in queries}

hits   = [r for r in results if r["hit"]]
misses = [r for r in results if not r["hit"]]
print(f"Hits  : {len(hits)}")
print(f"Misses: {len(misses)}")

print("\n── Sample hits ──────────────────────────────────────")
for r in hits[:2]:
    print(f"QID      : {r['qid']}")
    print(f"Query    : {query_text_map[r['qid']][:150]}")
    print(f"Retrieved: {r['retrieved']} (score={r['score']})")
    print(f"Text     : {corpus_text_map[r['retrieved']][:150]}")
    print()

print("\n── Sample misses ────────────────────────────────────")
for r in misses[:2]:
    print(f"QID         : {r['qid']}")
    print(f"Query       : {query_text_map[r['qid']][:150]}")
    print(f"Retrieved   : {r['retrieved']} (score={r['score']})")
    print(f"Retrieved T : {corpus_text_map[r['retrieved']][:150]}")
    print(f"N relevant  : {r['n_relevant']}")
    print()

Hits  : 3222
Misses: 1552

── Sample hits ──────────────────────────────────────
QID      : suspicious-document010364_query_0
Query    : In this study, we examine the three-dimensional inviscid forced Burgers equation coupled with the continuity equation. We derive the exact two-point c
Retrieved: source-document010364_para_7 (score=128.8948)
Text     : In this paper we consider the inviscid forced Burgers equation in 3–dimensions supplmented to continuity equation. We find the exact two–point correla

QID      : suspicious-document010364_query_1
Query    : Burgers' equation, which models the potential flow of an inviscid fluid, serves as an invaluable platform for exploring novel concepts and methodologi
Retrieved: source-document010364_para_3 (score=106.135)
Text     : On the other hand Burgers equation, which describes the potential flow of the fluid without pressure, provides a wonderful laboratory for testing new 


── Sample misses ────────────────────────────────────
QID        